# Hito 3 · Pronóstico horario de PM2.5 en Lima

**1ACC0057 Machine Learning · UPC**

El CSV original de SENAMHI usa **UTC** y el modelo trabaja en **America/Lima (UTC−5)**. La comprobación con OpenAQ encontró 112 coincidencias en 112 pares de cuatro estaciones dentro del redondeo de publicación. La unidad de PM2.5 es **µg/m³**.

El pronóstico estima la concentración una hora adelante. Con `RECALCULAR = False` se abren los resultados ya verificados. Para volver a entrenar, instala las dependencias de `requirements.txt` y apunta `DATOS` a los insumos originales indicados en el README.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

COLAB = "google.colab" in sys.modules or Path("/content").is_dir()
PREDETERMINADO = Path("/content/Viento-Lima") if COLAB else Path.cwd()
PROYECTO = Path(os.environ.get("PM25_PROYECTO", PREDETERMINADO)).resolve()
if COLAB and not PROYECTO.exists():
    subprocess.run(["git", "clone", "--branch", "develop", "--single-branch",
                    "https://github.com/KeniChris/Viento-Lima.git", str(PROYECTO)], check=True)
if not (PROYECTO / "codigo" / "preparar.py").is_file():
    raise FileNotFoundError("Fija PM25_PROYECTO a la carpeta que contiene codigo/")
sys.path.insert(0, str(PROYECTO))
DATOS = Path(os.environ.get("PM25_DATOS", PROYECTO / "datos"))
RESULTADOS = PROYECTO / "resultados"
RECALCULAR = False
EJECUTAR_HITO2 = False
HILOS = 4

## Verificación y preparación de los datos

OpenAQ publica fecha, hora, desfase y unidad de lecturas del SENAMHI. Por ejemplo, en San Borja el 15 de abril de 2024 se registraron **10,19 µg/m³** a las **06:00 UTC**, equivalentes a **01:00 en Lima**. La segunda interpretación se usa en la tabla como control de la verificación.

El pipeline conserva la marca UTC, convierte a Lima y cruza por estación y hora con la meteorología. Se preservan los faltantes de PM2.5 y se incorporan velocidad y dirección del viento IFS a 10 m. Al aplicar la convención meteorológica de procedencia, un viento del norte se representa como movimiento hacia el sur.

In [ ]:
if RECALCULAR:
    from codigo.actualizar_aire import verify_clock
    from codigo.preparar import prepare_dataset
    from codigo.viento import WIND_FILE, build_wind_file
    VIENTO = DATOS / WIND_FILE
    if not VIENTO.is_file():
        build_wind_file(VIENTO)
    verify_clock(DATOS / "aire_lima_completo.csv", RESULTADOS)
    matrix = prepare_dataset(DATOS, VIENTO, RESULTADOS)
display(pd.read_csv(RESULTADOS / "verificacion_horaria.csv"))
display(pd.read_csv(RESULTADOS / "ejemplos_hora_verificada.csv").head(8))
display(pd.read_csv(RESULTADOS / "auditoria_preparacion.csv"))

## Modelado y evaluación

Los pliegues expansivos validan en 2020, 2021 y 2022 y descartan las etiquetas que cruzan cada corte. La prueba histórica usa 56.035 pronósticos de 2023 a abril de 2024. La ablación mantiene los hiperparámetros de XGBoost y añade sucesivamente viento local, PM2.5 vecino y predictores de transporte.

Random Forest fue seleccionado por MAE medio entre estaciones en validación. En la prueba histórica, XGBoost base obtuvo el menor MAE y Random Forest el menor RMSE. Añadir transporte sobre el XGBoost con lecturas vecinas aumentó el MAE de 2,910 a 2,925 µg/m³.

In [ ]:
if RECALCULAR:
    from codigo.evaluar import evaluate_models
    evaluate_models(matrix, RESULTADOS, HILOS)
cv = pd.read_csv(RESULTADOS / "cv_modelos.csv")
test = pd.read_csv(RESULTADOS / "prueba_modelos.csv")
display(cv[["modelo", "MAE_macro_cv", "DE_MAE_macro_cv", "RMSE_cv"]].round(4))
display(test[["modelo", "n", "MAE", "MAE_macro", "RMSE", "R2",
              "mae_altos", "sesgo_altos", "sensibilidad_altos"]].round(4))
display(pd.read_csv(RESULTADOS / "diferencias_prueba.csv").round(4))
display(pd.read_csv(RESULTADOS / "prueba_por_estacion.csv")
        [["modelo", "ESTACION", "n", "MAE", "mae_altos"]].round(4))

## Prueba posterior al corte de SENAMHI

Las lecturas de OpenAQ posteriores al 30 de abril de 2024 se evalúan con los modelos ajustados con datos anteriores a 2023. La cobertura y los códigos de respuesta del archivo diario documentan los intervalos disponibles.

In [ ]:
if RECALCULAR:
    from codigo.actualizar_aire import EXTENSION_FILE, download_extension
    from codigo.validar_extension import evaluate_extension
    if not (DATOS / EXTENSION_FILE).is_file():
        download_extension(DATOS, RESULTADOS)
    evaluate_extension(DATOS, RESULTADOS, jobs=HILOS)
display(pd.read_csv(RESULTADOS / "cobertura_openaq_2024.csv"))
display(pd.read_csv(RESULTADOS / "acceso_openaq_2024.csv"))
display(pd.read_csv(RESULTADOS / "prueba_extension_2024.csv").round(4))
display(pd.read_csv(RESULTADOS / "diferencia_transporte_extension_2024.csv").round(4))

## Integración con PostgreSQL

La prueba local leyó las siete entradas de una hora, escribió siete pronósticos para la hora siguiente y comprobó la repetición sin duplicados. El contrato SQL y el adaptador están en `codigo/`. La instancia común del módulo de IA requiere una comprobación independiente.

In [ ]:
print((RESULTADOS / "conexion_postgresql.json").read_text(encoding="utf-8"))
display(pd.read_csv(RESULTADOS / "conexion_pronosticos.csv").round(4))

## Fuentes

- [SENAMHI · Monitoreo de contaminantes de Lima Metropolitana](https://www.datosabiertos.gob.pe/dataset/monitoreo-de-los-contaminantes-del-aire-en-lima-metropolitana-servicio-nacional-de)
- [OpenAQ · Formato de datos y desfase horario](https://docs.openaq.org/aws/about)
- [OpenAQ · San Borja](https://explore.openaq.org/locations/2402)
- [Open-Meteo · Viento histórico de ECMWF IFS](https://open-meteo.com/en/docs/historical-weather-api)
- [Viento-Lima · Rama develop](https://github.com/KeniChris/Viento-Lima/tree/develop)